# Day 1 · Lab A — Make the Modeling Decision
**Domain:** telco **Customer Service / Sales** churn.

**Goal:** this lab is about *judgement*, not just accuracy. You'll train three models,
compare them with honest metrics, tune the decision threshold to a **business budget**,
and write a short justification for your choice.

> Dataset: IBM *Telco Customer Churn* (see `../datasets/README.md`).

In [1]:
"""
This cell loads and prepares the customer churn data for modeling.
Think of it like:
  1. Reading a customer database from a CSV file
  2. Cleaning up messy data (fixing invalid numbers, removing incomplete records)
  3. Labeling what we want to predict (Churn: Yes/No → 1/0)
  4. Splitting into a training set (learn patterns) and test set (measure accuracy)
"""

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, precision_recall_curve)

# STEP 1: Locate the datasets directory
# Try multiple possible paths to find where the data is stored
DATA = next((p for p in ('datasets', '../datasets', '../../datasets') if os.path.isdir(p)), None)
# If none of the paths exist, raise an error with helpful message
if DATA is None:
    raise FileNotFoundError("Could not find a datasets directory. Tried: datasets, ../datasets, ../../datasets")

# STEP 2: Load the CSV file into a pandas DataFrame
df = pd.read_csv(os.path.join(DATA, 'telco_customer_churn.csv'))

df.head(10)  # Display the first few rows of the DataFrame to check the data

# STEP 3: Clean the TotalCharges column
# Convert to numeric type; if a value can't be converted, mark it as NaN (Not a Number)
# errors='coerce' is safer than errors='raise' because it doesn't crash on bad data
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df.head(10)  # Display the first few rows again to see the changes in TotalCharges
# STEP 4: Remove rows with missing values and drop unnecessary columns
# dropna() removes any row that has at least one NaN value
# drop(columns=['customerID']) removes the ID column (not useful for prediction)
df = df.dropna().drop(columns=['customerID'])

# STEP 5: Create the target variable (what we're trying to predict)
# Check if Churn == 'Yes', convert to 1 (True) or 0 (False)
# astype(int) converts boolean to integer (1 or 0)
y = (df['Churn'] == 'Yes').astype(int)

# STEP 6: Create the feature matrix (inputs to the model)
# Drop the 'Churn' column since that's now our target (y)
# pd.get_dummies() converts categorical variables to binary columns (one-hot encoding)
# drop_first=True removes one dummy variable to avoid multicollinearity
X = pd.get_dummies(df.drop(columns=['Churn']), drop_first=True)

# STEP 7: Split data into training and testing sets
# test_size=0.2 means 80% train, 20% test
# random_state=7 ensures reproducibility (same split every time)
# stratify=y ensures both train and test have similar churn rates (important for imbalanced data)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=7, stratify=y)

# STEP 8: Print dataset statistics
# Calculate and print the churn rate (% of positive class) in training data
# round(..., 3) limits to 3 decimal places
print('positive (churn) rate:', round(ytr.mean(), 3))

positive (churn) rate: 0.266


## 1. Train three candidates
A trivial **baseline**, an interpretable **logistic regression**, and a flexible **random forest**.

### Exercise 1 — fit the three models
Baseline = `DummyClassifier(strategy='most_frequent')`; logistic in a scaling pipeline;
random forest with 200 trees.

In [3]:
"""
We're training 3 different models to predict which customers will churn.
Think of it like interviewing 3 different consultants who each use their own strategy:
  1. Baseline Consultant: Always says "no churn" (lazy but gives us a floor to beat)
  2. Logistic Consultant: Uses simple linear reasoning (easy to understand)
  3. Random Forest Consultant: Asks many questions and combines many simple rules (more complex)
Each consultant learns from historical data and will give us predictions.
"""

# EXERCISE 1: Train three candidate models

# Model 1: BASELINE - Always predicts the most frequent class (no churn)
# This is our "do nothing" baseline to compare against
# It will have high accuracy but 0 recall (catches no churners)
baseline = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)

# Model 2: LOGISTIC REGRESSION - Interpretable linear model
# Pipeline applies StandardScaler first, then LogisticRegression
# StandardScaler normalizes features (subtracts mean, divides by std dev)
# This improves convergence and is important for logistic regression
# max_iter=1000 allows up to 1000 iterations to find optimal weights
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=100)).fit(Xtr, ytr)

# Model 3: RANDOM FOREST - Ensemble of 200 decision trees
# n_estimators=200 trains 200 different trees and averages their predictions
# More trees = better but slower; 200 is a good balance
# random_state=7 ensures reproducibility (same trees every time we run)
rf = RandomForestClassifier(n_estimators=200, random_state=7).fit(Xtr, ytr)

# Store all models in a dictionary for easy access later
# This allows us to loop through all models in the evaluation step
models = {'baseline': baseline, 'logistic': logreg, 'random_forest': rf}

## 2. Compare with honest metrics
Accuracy alone is misleading on imbalanced data. Compare precision / recall / F1 / ROC-AUC.

### Exercise 2 — complete the metrics row
Fill in recall and ROC-AUC (use `predict_proba` for AUC; the baseline has no useful proba).

In [ ]:
"""
This function compares all three models using fair metrics.
Think of it like:
  - accuracy: Out of 100 predictions, how many were correct? (Tricky: not always best metric!)
  - precision: If we flag someone, is it actually a churner? (Quality of our flags)
  - recall: Of all real churners, how many did we catch? (Are we missing churners?)
  - f1: A balanced combination of precision and recall
  - roc_auc: Overall model quality (1.0 = perfect, 0.5 = random guessing)
"""

def evaluate(name, model):
    # Get hard predictions (0 or 1) from the model on test data
    yhat = model.predict(Xte)
    
    # Try to get probability predictions (for ROC-AUC calculation)
    try:
        # Get probability that each sample belongs to class 1 (churn)
        # [:, 1] selects the second column (probability of churn, not no-churn)
        proba = model.predict_proba(Xte)[:, 1]
        # Calculate ROC-AUC: measures model's ability to distinguish classes at all thresholds
        # ROC-AUC ranges from 0 to 1 (1.0 = perfect prediction)
        auc = roc_auc_score(yte, proba)
    except (AttributeError, ValueError):
        # Some models (like DummyClassifier) don't have predict_proba method
        # Set AUC to NaN so we can still see other metrics
        auc = float('nan')
    
    # Return dictionary with all metrics for this model
    return {
        'model': name,  # Model name
        'accuracy': accuracy_score(yte, yhat),  # % of correct predictions (misleading for imbalanced data!)
        'precision': precision_score(yte, yhat, zero_division=0),  # Of flagged customers, how many actually churned?
        'recall': recall_score(yte, yhat, zero_division=0),  # Of actual churners, how many did we catch?
        'f1': f1_score(yte, yhat, zero_division=0),  # Harmonic mean of precision and recall (good single metric)
        'roc_auc': auc  # Area under ROC curve (considers all probability thresholds)
    }

# Evaluate each model and create a comparison table
# List comprehension: for each model, calculate all metrics
table = pd.DataFrame([evaluate(n, m) for n, m in models.items()]).round(3)
# Print as formatted table without index column
print(table.to_string(index=False))

        model  accuracy  precision  recall    f1  roc_auc
     baseline     0.734      0.000   0.000 0.000    0.500
     logistic     0.809      0.668   0.559 0.608    0.856
random_forest     0.810      0.692   0.516 0.591    0.846


Notice the baseline's ~73% accuracy with **0 recall** — useless, but 'accurate'.

### Gradient boosting — the tabular default
On tabular data, **gradient-boosted trees** are usually the strongest *classical* option, and
still beat deep nets on most tabular benchmarks. sklearn's `HistGradientBoostingClassifier`
needs no extra install — add it to the comparison.

In [ ]:
"""
Gradient Boosting is like asking a group of consultants to make predictions one by one.
- First consultant makes a guess
- Second consultant learns from first consultant's mistakes and corrects them
- Third consultant corrects the second one's mistakes
- And so on... each one learning from previous mistakes
This "ensemble" approach is very powerful and often wins competitions!
"""

# Train Histogram Gradient Boosting model
# This is often the best performer on tabular data (like spreadsheets)
# random_state=7 ensures reproducibility (same model every time)
hgb = HistGradientBoostingClassifier(random_state=7).fit(Xtr, ytr)

# Add it to our models dictionary
models['hist_gb'] = hgb

# Evaluate and print metrics for this new model
row = pd.DataFrame([evaluate('hist_gb', hgb)]).round(3)
print(row.to_string(index=False))

  model  accuracy  precision  recall    f1  roc_auc
hist_gb     0.809      0.669   0.556 0.607    0.855


## 3. Tune the decision threshold to a budget
The model outputs a probability; **you** choose the cut-off. Suppose the retention team can
only contact **~30% of customers**. Find the threshold that flags ~30% and see what recall
(churners caught) you get.

### Exercise 3 — pick the threshold for the budget

In [ ]:
"""
SUMMARY FOR NON-ML ENGINEERS:
The model gives us a probability (0-100%) that each customer will churn.
We don't use that probability directly - we pick a cutoff point (threshold).
Think of it like: "If the probability is above X%, flag them for retention outreach."
Here we tune this threshold based on business constraints:
- Retention team can only contact 30% of customers
- What's the best threshold to flag the top 30%?
- How many actual churners do we catch? (recall)
- How many of our flagged customers actually churn? (precision)
"""

# Get predicted probabilities for churn (values between 0 and 1)
# These represent the model's confidence that each customer will churn
proba = logreg.predict_proba(Xte)[:, 1]

# Business constraint: retention team can only contact 30% of customers
budget = 0.30

# Find the probability threshold that flags the top 30% of customers
# np.quantile(proba, 1 - budget) finds the 70th percentile (top 30%)
# Customers with probability >= this threshold are flagged for retention
thr = np.quantile(proba, 1 - budget)

# Create binary flags: True if probability >= threshold, False otherwise
flagged = proba >= thr

# METRIC 1: Recall (sensitivity) - Of all actual churners, what % did we catch?
# High recall = we catch most churners, but may flag many non-churners
print('recall at this budget:', round(recall_score(yte, flagged), 3))

# METRIC 2: Precision - Of all customers we flagged, what % actually churned?
# High precision = our flags are accurate, but we may miss some churners
print('precision at this budget:', round(precision_score(yte, flagged), 3))

recall at this budget: 0.679
precision at this budget: 0.602
